# M1.2: Your first PyPSA network — a Croatia-like system on one bus

## Objectives

At the end of this notebook you should be able to:

- explain PyPSA's data model: a `Network` with snapshots and components (buses, carriers, loads, generators)
- tell **static** attributes (`n.generators`) from **time-varying** ones (`n.generators_t`)
- build and optimise a one-week dispatch model and read dispatch, prices and total cost
- validate the result yourself: energy balance, capacity limits, cost, and prices from the merit order
- find the linear program PyPSA built (`n.model`) and connect its nodal balance to Kirchhoff's current law

This is the start of the **running model** that grows through the whole course. Today it is a single bus — the whole country as one node, no grid. In Module 03 you will open up its LP completely, in Module 04 it becomes a network, in Module 05 it starts building capacity.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)   # keep outputs readable
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
print("PyPSA", pypsa.__version__)

PyPSA 1.4.0


### PyPSA's data model

A PyPSA **`Network`** holds everything as pandas tables:

| Concept | What it is | Where it lives |
|---|---|---|
| **snapshots** | the time steps (here: hours) | `n.snapshots` |
| **Bus** | a node where energy must balance (KCL!) | `n.buses` |
| **Carrier** | an energy type, with CO₂ intensity | `n.carriers` |
| **Load** | demand at a bus, MW | `n.loads`, `n.loads_t.p_set` |
| **Generator** | supply with capacity `p_nom` (MW), `marginal_cost` (€/MWh), availability `p_max_pu` | `n.generators`, `n.generators_t` |

**Static** attributes (one value per component) sit in `n.generators`; **time-varying** ones (one value per snapshot) in `n.generators_t`. Availability `p_max_pu` is *per unit* of capacity: 0.3 means 30 % of `p_nom` is available in that hour.

### Input data: a stylised week

The function below creates a **synthetic** winter week: load shaped like Croatian demand (~2,100 MW average), plus wind and solar availability profiles. It is clearly not measured data — real load comes from ENTSO-E (M1.1, your exercise), real wind and solar profiles in M1.4. Starting from known synthetic inputs lets you check every result by hand.

In [2]:
def synthetic_profiles(start: str = "2025-01-13", days: int = 7, seed: int = 1) -> pd.DataFrame:
    """SYNTHETIC hourly profiles (UTC): load in MW, wind and solar as per-unit availability.

    Shapes are plausible for a Croatia-sized winter week but are NOT measured data.
    Real load comes from ENTSO-E (M1.1), real wind/solar profiles in M1.4.
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, periods=days * 24, freq="h")          # naive timestamps = UTC
    local = idx.tz_localize("UTC").tz_convert("Europe/Zagreb")
    hour, weekend = local.hour.to_numpy(), (local.dayofweek >= 5)
    load = (2150 + 380 * np.sin((hour - 7) / 24 * 2 * np.pi) - 200 * weekend
            + rng.normal(0, 40, len(idx)))
    wind = np.empty(len(idx))
    wind[0] = 0.35
    for t in range(1, len(idx)):                                        # slowly varying wind
        wind[t] = np.clip(0.9 * wind[t - 1] + 0.1 * 0.35 + rng.normal(0, 0.06), 0.02, 0.95)
    solar = np.clip(np.sin(np.pi * (hour - 7.5) / 9), 0, None) * rng.uniform(0.3, 0.8, len(idx))
    return pd.DataFrame({"load_MW": load.round(1), "wind_pu": wind.round(3),
                         "solar_pu": solar.round(3)}, index=idx)


profiles = synthetic_profiles()
profiles.describe().round(2).T

          count     mean     std      min     25%      50%      75%      max
load_MW   168.0  2090.00  284.78  1522.00  1834.0  2084.05  2343.20  2600.10
wind_pu   168.0     0.27    0.09     0.06     0.2     0.27     0.32     0.54
solar_pu  168.0     0.14    0.22     0.00     0.0     0.00     0.29     0.77

Wind averages 27 % of capacity this week and solar 14 % (zero at night, low winter sun). These **capacity factors** are what make renewables so different from conventional plants: 1,000 MW of wind delivers far less than 1,000 MW on average.

### Building the network

The system has one bus, `HR`, and a stylised mix with **round numbers inspired by the Croatian system — not official capacity data**: a share of a nuclear plant, run-of-river hydro, wind, solar, a coal unit, gas plants, and imports.

Two modelling choices deserve a flag right away:
- **Imports as a generator** at 120 €/MWh: Croatia imports a lot of electricity, and treating neighbours as a supplier at a fixed price is a common first simplification. Module 04 replaces it with real interconnectors.
- **Hydro as flat run-of-river** at 45 % availability: real reservoirs store water and decide *when* to generate. That is storage optimisation (Module 03).

In [3]:
def build_network(profiles: pd.DataFrame, with_imports: bool = True) -> pypsa.Network:
    """Stylised single-bus Croatia-like system. Round numbers for teaching, not official data."""
    n = pypsa.Network()
    n.set_snapshots(profiles.index)
    n.add("Carrier", "AC")
    n.add("Bus", "HR", carrier="AC")

    # carriers: co2_emissions in t CO2 per MWh of fuel (thermal)
    for name, co2 in [("nuclear", 0.0), ("hydro", 0.0), ("wind", 0.0), ("solar", 0.0),
                      ("coal", 0.34), ("gas", 0.20), ("imports", 0.0)]:
        n.add("Carrier", name, co2_emissions=co2)

    n.add("Load", "demand", bus="HR", carrier="AC", p_set=profiles["load_MW"])
    #     name           carrier    p_nom MW  marginal cost EUR/MWh  efficiency
    plants = [("nuclear_share", "nuclear", 350, 10, 0.33),
              ("hydro_ror", "hydro", 1500, 2, 1.0),
              ("wind", "wind", 1000, 0, 1.0),
              ("solar", "solar", 800, 0, 1.0),
              ("coal", "coal", 210, 95, 0.38),
              ("gas_ccgt", "gas", 700, 110, 0.58),
              ("gas_ocgt", "gas", 300, 160, 0.38)]
    for name, carrier, p_nom, mc, eff in plants:
        n.add("Generator", name, bus="HR", carrier=carrier, p_nom=p_nom,
              marginal_cost=mc, efficiency=eff)
    n.generators_t.p_max_pu["hydro_ror"] = 0.45      # run-of-river, flat (simplification)
    n.generators_t.p_max_pu["wind"] = profiles["wind_pu"]
    n.generators_t.p_max_pu["solar"] = profiles["solar_pu"]
    if with_imports:
        n.add("Generator", "imports", bus="HR", carrier="imports", p_nom=1500, marginal_cost=120)
    return n


n = build_network(profiles)
n.generators[["carrier", "p_nom", "marginal_cost", "efficiency"]]

               carrier   p_nom  marginal_cost  efficiency
name                                                     
nuclear_share  nuclear   350.0           10.0        0.33
hydro_ror        hydro  1500.0            2.0        1.00
wind              wind  1000.0            0.0        1.00
solar            solar   800.0            0.0        1.00
coal              coal   210.0           95.0        0.38
gas_ccgt           gas   700.0          110.0        0.58
gas_ocgt           gas   300.0          160.0        0.38
imports        imports  1500.0          120.0        1.00

In [4]:
n.generators_t.p_max_pu.head(3).round(2)

name                 hydro_ror  wind  solar
snapshot                                   
2025-01-13 00:00:00       0.45  0.35    0.0
2025-01-13 01:00:00       0.45  0.31    0.0
2025-01-13 02:00:00       0.45  0.31    0.0

Notice that only hydro, wind and solar appear in `n.generators_t.p_max_pu`: every other generator uses its static default of 1.0 (fully available). PyPSA only stores time series where you set them.

### Optimising the dispatch

`n.optimize()` builds a linear program — minimise total generation cost, subject to supply = demand at every bus in every hour and every generator within its available capacity — and hands it to HiGHS.

In [5]:
status = n.optimize(solver_name="highs", log_to_console=False)
status

('ok', 'optimal')

In [6]:
dispatch = n.generators_t.p
energy_GWh = (dispatch.sum() / 1e3).round(1).rename("energy_GWh")
energy_GWh.to_frame().assign(capacity_factor_pct=(100 * dispatch.mean() / n.generators.p_nom).round(1))

               energy_GWh  capacity_factor_pct
name                                          
nuclear_share        58.8                100.0
hydro_ror           113.4                 45.0
wind                 45.1                 26.8
solar                18.9                 14.0
coal                 34.9                 98.8
gas_ccgt             77.0                 65.5
gas_ocgt              0.0                  0.0
imports               3.1                  1.2

The cheap plants (nuclear, hydro, wind, solar) run whenever they are available. Coal and CCGT fill most of the rest. Imports set the price in many peak hours but deliver little energy — they top up the peaks. The OCGT at 160 €/MWh never runs — importing at 120 €/MWh is always cheaper. That is the **merit order**, found by an optimiser.

In [7]:
fig, ax = plt.subplots(figsize=(10, 3.5))
order = ["nuclear_share", "hydro_ror", "wind", "solar", "coal", "gas_ccgt", "imports"]
ax.stackplot(dispatch.index, [dispatch[g] for g in order], labels=order, alpha=0.85)
ax.plot(n.loads_t.p_set.index, n.loads_t.p_set["demand"], color="black", lw=1.2, label="load")
ax.set_ylabel("MW"); ax.set_title("Dispatch, stylised Croatia-like week (synthetic data)")
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.show()

### Prices and total cost

In [8]:
price = n.buses_t.marginal_price["HR"]
print(f"total cost: {n.objective:,.0f} EUR for {n.loads_t.p_set['demand'].sum():,.0f} MWh")
print(f"average system cost: {n.objective / n.loads_t.p_set['demand'].sum():.1f} EUR/MWh")
price.value_counts().sort_index().rename("hours at this price")

total cost: 12,973,492 EUR for 351,119 MWh
average system cost: 36.9 EUR/MWh


HR
95.0       5
110.0    133
120.0     30
Name: hours at this price, dtype: int64

The price is always one of three numbers: 95 (coal), 110 (CCGT) or 120 €/MWh (imports). In every hour it equals the marginal cost of the **most expensive unit running** — the plant that would serve one more MWh. The average generation cost (about 37 €/MWh) is far below every price: most energy comes from plants that cost much less to run than the price they are paid. That gap is their margin — the heart of Module 02.

### Validate the result yourself

PyPSA reporting "optimal" only means the solver solved the problem it was given. Before you trust any number, check it against physics and economics — the habit from EE-1.1 and M1.1.

In [9]:
load = n.loads_t.p_set["demand"]
available = n.get_switchable_as_dense("Generator", "p_max_pu") * n.generators.p_nom

balance_gap = (dispatch.sum(axis=1) - load).abs().max()
over_capacity = (dispatch - available).max().max()
cost_by_hand = (dispatch * n.generators.marginal_cost).sum().sum()

assert balance_gap < 1e-6, "supply must equal demand every hour"
assert over_capacity < 1e-6, "no generator above its available capacity"
assert np.isclose(cost_by_hand, n.objective), "objective must equal sum of p x marginal cost"
print(f"max balance gap {balance_gap:.2e} MW | max over-capacity {over_capacity:.2e} MW | cost check OK")

max balance gap 4.55e-13 MW | max over-capacity 0.00e+00 MW | cost check OK


`get_switchable_as_dense` returns every generator's availability per hour, filling in the static default where no time series exists — useful whenever you need the full picture.

Now the economics: for a single bus without storage or ramping, each hour is independent, so the price must equal the marginal cost of the last plant in the **merit order**. Reproduce the price for every hour without the solver:

In [10]:
def merit_order_price(available_MW: pd.Series, marginal_cost: pd.Series, demand_MW: float) -> float:
    """Marginal cost of the plant that serves the last MW of demand."""
    order = marginal_cost.sort_values().index
    cumulative = available_MW[order].cumsum()
    return float(marginal_cost[order][cumulative >= demand_MW - 1e-9].iloc[0])

mo_price = pd.Series([merit_order_price(available.loc[t], n.generators.marginal_cost, load[t])
                      for t in n.snapshots], index=n.snapshots)
mismatch = (mo_price != price)
print(f"hours where solver price != merit-order price: {mismatch.sum()} of {len(price)}")

hours where solver price != merit-order price: 0 of 168


The solver and the merit order agree in every hour. You now know *why* the prices are what they are — and you have a test that would catch a wrong marginal cost or a missing availability profile.

### Emissions: why efficiency matters

Carrier CO₂ intensities are per MWh of **fuel** (thermal). A plant burns more fuel than it produces electricity, so its emissions per MWh of **electricity** are `co2_emissions / efficiency`: coal 0.34 / 0.38 ≈ 0.89 t/MWh, CCGT 0.20 / 0.58 ≈ 0.34 t/MWh.

In [11]:
co2_per_MWh_el = (n.generators.carrier.map(n.carriers.co2_emissions) / n.generators.efficiency)
emissions_t = (dispatch * co2_per_MWh_el).sum()
emissions_t[emissions_t > 0].round(0).rename("t CO2 this week")

name
coal        31184.0
gas_ccgt    26558.0
Name: t CO2 this week, dtype: float64

Imports show zero emissions — not because imported electricity is clean, but because they fall outside our **accounting boundary**. Every model has such boundaries; state them whenever you report emissions.

### Inside the box: the LP PyPSA built

PyPSA translates the network into a Linopy model. Look at the constraint names, then at the nodal balance for the first hour — Kirchhoff's current law from EE-1.1, written in MW: everything injected at the bus equals the load.

In [12]:
print(n.model.constraints)
print(n.model.constraints["Bus-nodal_balance"].sel(snapshot=n.snapshots[0]))

linopy.model.Constraints
------------------------
 * Generator-fix-p-lower (snapshot, name)
 * Generator-fix-p-upper (snapshot, name)
 * Bus-nodal_balance (snapshot, name)

Constraint `Bus-nodal_balance` [name: 1]:
-----------------------------------------
[HR]: +1 Generator-p[2025-01-13 00:00:00, nuclear_share] + 1 Generator-p[2025-01-13 00:00:00, hydro_ror] + 1 Generator-p[2025-01-13 00:00:00, wind] ... +1 Generator-p[2025-01-13 00:00:00, gas_ccgt] + 1 Generator-p[2025-01-13 00:00:00, gas_ocgt] + 1 Generator-p[2025-01-13 00:00:00, imports] = 1783.8


The price you read from `n.buses_t.marginal_price` is the **dual value** of exactly this constraint: how much the total cost rises if demand in that hour increases by 1 MW. Module 03 derives this properly; for now, the merit-order check above showed it gives the right answer.

## Check your understanding

**Reading the model**

1. Before running anything: in an hour with 2,500 MW load, 300 MW wind and no sun, which plant sets the price? Check your reasoning against `price`.
2. Why does the OCGT never run? Change its marginal cost to 115 €/MWh, re-optimise, and predict first how the price distribution changes.

**Changing the system**

1. Double the solar capacity. Predict the effect on total cost, emissions and the number of hours at 120 €/MWh, then check.
2. Build the network with `with_imports=False`. Which plant now sets the price at the evening peaks? Then also take the coal unit out for maintenance (`n.generators.loc["coal", "p_nom"] = 0`) and optimise again. What happens, and why? (You will fix this in the exercise.)

**Thinking ahead**

1. The price was the same for all generators in each hour, although their costs differ. Who earns money in this market, and how much? (Module 02.)
2. Which result in this notebook would change if hydro could store water and choose when to generate?